In [1]:
import requests
import pandas as pd 
from bs4 import BeautifulSoup as BS
import matplotlib_inline as mp
from urllib.parse import urljoin

In [2]:
url = url = "https://www.redbubble.com/shop/crying"
params = {"country": "IN", "deviceType": "desktop", "enableRedirects": "true", "locale": "en", "pageSize": 82, "ref": "search_box"}
headers = {"User-Agent": ("Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7)" "AppleWebKit/537.36 (KHTML, like Gecko)" "Chrome/127.0.0.0 Safari/537.36")}

In [3]:
products_data = []
products = []

In [4]:
for page_no in range (1, 6):

    params["page"] = page_no
    print("Scrapping page:", page_no)
    reponse = requests.get(url, params = params ,headers = headers)

    print("Status    :",reponse.status_code)

    if (reponse.status_code != 200): 
        print("book not found")
        continue

    soup = BS(reponse.content, "html.parser")
    # with open(f"redbubble{page_no}.html", "w", encoding="utf-8") as f: f.write(soup.prettify())
    links = soup.find_all("a", href = True)

    products_per_page = []

    for link in links:
        href = link["href"]
        if "/i/" in href:
            product_url = urljoin("https://www.redbubble.com", href)
            if product_url not in products_per_page:
                products_per_page.append(link)

    products.extend(products_per_page)
    print("products found:", len(products))

Scrapping page: 1
Status    : 200
products found: 90
Scrapping page: 2
Status    : 200
products found: 180
Scrapping page: 3
Status    : 200
products found: 270
Scrapping page: 4
Status    : 200
products found: 360
Scrapping page: 5
Status    : 200
products found: 450


In [5]:
for product in products:

    name = product.find("span", class_=lambda x: x and "SearchResultCard_title__" in x)
    name = name.get_text(strip=True) if name else None

    author = None

    if name:
        title_block = product.find("span",class_=lambda x: x and "SearchResultCard_title__" in x).parent
        author_tag = title_block.find("div").find("span")
        if author_tag: author = author_tag.get_text(strip=True)

    
    price = product.find("span", attrs={"data-testid": "line-item-price-price"})
    price = price.get_text(strip=True) if price else None

    original = product.find("span", attrs={"data-testid": "line-item-undiscounted-price"})
    original = original.find("del").get_text(strip=True)
    products_data.append({"name": name, "author": author, "price": price, "original price": original})


In [6]:
df = pd.DataFrame(products_data)
df.head()

,name,author,price,original price
0,confused loading cat sticker Sticker,The Kiwi Store,$2.51,$2.78
1,Everything’s good cat Sticker,Ivanna Baca,$2.51,$2.78
2,Did The Thing Anyway Pink Sticker,Me And The Moon,$3.34,$3.71
3,Hand Drawn Monsters Crying Sketch Classic T-Shirt,ValueShop3,$17.54,$25.06
4,Crying Dawson Sticker,aelmz,$2.72,$3.02


In [7]:
df.to_csv("meme_products.csv",index=False,encoding="utf-8-sig")